# File

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv("ML_Ready_Extended_Geography.csv") # ETO YUNG GINAMIT KONG CSV YA CHINECK Q NA
X = df.drop('HIV_Status', axis=1)
y = df['HIV_Status']

# Splitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
df.head(10)

,Sex,Age_Group,Transmission,Healthcare_Access_Friction,Civil_Status,OFW_Status,Chemsex_Engagement,Alcohol_Sex_Risk,PrEP_Awareness,Transactional_Sex,STI_BBV_CoInfection_Any,HIV_Status
0,Female,<15,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
1,Male,15-24,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
2,Male,15-24,Male to Male Sex,2,Single,No,No,No,Yes,No TS,Yes,Reactive
3,Male,15-24,Male to Male Sex,2,Single,No,No,No,No,No TS,Yes,Reactive
4,Male,15-24,Male to Male Sex,2,Single,No,Yes,No,Yes,No TS,No,Reactive
5,Male,15-24,Male to Male/Female Sex,2,Single,No,No,Yes,Yes,Both,No,Reactive
6,Male,25-34,Male to Female Sex,2,Common-Law,No,No,No,No,No TS,No,Reactive
7,Male,25-34,Male to Male Sex,2,Single,No,No,No,Yes,No TS,No,Reactive
8,Male,25-34,Male to Male Sex,2,Single,No,No,No,No,No TS,No,Reactive
9,Male,25-34,Male to Male Sex,2,Single,No,No,Yes,Yes,Paid for sex,Yes,Reactive


# Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# list num and cat
numeric_features = []
categorical_features = ['Sex','Age_Group','Transmission',
                        'Healthcare_Access_Friction','Civil_Status','OFW_Status','Chemsex_Engagement',
                        'Alcohol_Sex_Risk','PrEP_Awareness','Transactional_Sex','STI_BBV_CoInfection_Any']

# 2. proprocessor
preprocessor = ColumnTransformer(
    transformers=[
        # Apply StandardScaler to numeric columns
        ('num', StandardScaler(), numeric_features),

        # Apply OneHotEncoder to categorical columns
        # handle_unknown='ignore' ensures the code won't crash if X_test has a category not seen in X_train
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # Keep any other columns not listed above (or use 'drop')
)

# 3. Fit and Transform X_train
# CRITICAL: We fit ONLY on X_train to learn the mean/std and categories.
# Then we transform both train and test using those learned values.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# convert into df
X# Convert into DataFrame AND preserve the original index
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index
)

# Dealing with y columns
mapping = {'Non-Reactive': 0, 'Reactive': 1}

# y_train and y_test already keep their indices, but it's good practice
# to ensure it's a Series or 1D array for Scikit-Learn classifiers
y_train_processed = y_train.map(mapping)
y_test_processed = y_test.map(mapping)

 # MODEL

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score, average_precision_score
import xgboost as xgb
import numpy as np
# 1. Calculate the scale_pos_weight for your 1:10 imbalance
neg_count = (y_train_processed == 0).sum()
pos_count = (y_train_processed == 1).sum()
spw = neg_count / pos_count
print(f"Calculated scale_pos_weight for XGBoost: {spw:.2f}")

# 2. Initialize Base XGBoost
xgb_base = xgb.XGBClassifier(
    scale_pos_weight=spw,
    random_state=42,
    eval_metric='aucpr', # Tell it to optimize for Precision-Recall Area!
    n_jobs=-1
)

# 3. Define the Hyperparameter Grid
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],         # Tree depth (lower prevents overfitting)
    'learning_rate': [0.01, 0.1]    # How fast it learns from mistakes
}

# 4. Setup Grid Search
grid_search_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    scoring='average_precision',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# 5. Train the Model!
print("\nTraining XGBoost...")
grid_search_xgb.fit(X_train_processed, y_train_processed)

best_xgb_model = grid_search_xgb.best_estimator_

# 6. Make Predictions
y_pred_xgb = best_xgb_model.predict(X_test_processed)
y_probs_xgb = best_xgb_model.predict_proba(X_test_processed)[:, 1]

# 7. Evaluate
print("\n--- XGBoost Final Report ---")
print(f"Best Parameters: {grid_search_xgb.best_params_}")
print(classification_report(y_test_processed, y_pred_xgb, target_names=['Non Reactive', 'Reactive']))
print(f"XGB F1-Score: {f1_score(y_test_processed, y_pred_xgb):.4f}")
print(f"XGB AUPRC: {average_precision_score(y_test_processed, y_probs_xgb):.4f}")

# Threshold Moving 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, 
    precision_score, 
    recall_score,
    precision_recall_curve,      
    average_precision_score       
)

# 1. Get the actual probabilities from your winning XGBoost model
xgb_probs = best_xgb_model.predict_proba(X_test_processed)[:, 1]

# 2. Test every threshold from 0.01 to 0.99
thresholds = np.arange(0.01, 1.0, 0.01)
f1_scores = []

for t in thresholds:
    # If probability is greater than 't', predict 1, else 0
    custom_preds = (xgb_probs >= t).astype(int)
    f1 = f1_score(y_test_processed, custom_preds)
    f1_scores.append(f1)

# 3. Find the exact threshold that maximizes F1
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Default (50%) F1-Score: {f1_score(y_test_processed, best_xgb_model.predict(X_test_processed)):.4f}")
print(f"Best Custom Threshold: {best_threshold:.2f} (or {best_threshold*100:.0f}%)")
print(f"New Maximum F1-Score: {best_f1:.4f}")

# 4. See what the new Precision and Recall look like!
optimal_preds = (xgb_probs >= best_threshold).astype(int)
print(f"New Precision: {precision_score(y_test_processed, optimal_preds):.4f}")
print(f"New Recall: {recall_score(y_test_processed, optimal_preds):.4f}")

In [ ]:
# 1. Calculate Precision-Recall Curve data
precision, recall, pr_thresholds = precision_recall_curve(y_test_processed, xgb_probs)
auprc = average_precision_score(y_test_processed, xgb_probs)

# 2. Create the Figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# --- PLOT 1: Threshold vs Metrics ---
ax1.plot(thresholds, f1_scores, label='F1-Score', color='#39C5BB', linewidth=4)
# We calculate precision/recall lists for the same 0.01-0.99 thresholds for this plot
p_list = [precision_score(y_test_processed, (xgb_probs >= t).astype(int), zero_division=0) for t in thresholds]
r_list = [recall_score(y_test_processed, (xgb_probs >= t).astype(int)) for t in thresholds]

ax1.plot(thresholds, p_list, label='Precision', color='#FFBACC', linestyle='--', alpha=0.8)
ax1.plot(thresholds, r_list, label='Recall', color='#FFCC11', linestyle='--', alpha=0.8)

# Highlight the Best Threshold
ax1.axvline(best_threshold, color='#DD4444', linestyle=':', label=f'Best Threshold: {best_threshold:.2f}')
ax1.fill_between(thresholds, 0, f1_scores, color='#39C5BB', alpha=0.1)

ax1.set_title(f"XGBoost Threshold Optimization\nMax F1: {best_f1:.4f}", fontsize=14)
ax1.set_xlabel("Probability Threshold")
ax1.set_ylabel("Score")
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# --- PLOT 2: Precision-Recall Curve ---
ax2.plot(recall, precision, color='#39C5BB', linewidth=4, label=f'PR Curve (AUPRC = {auprc:.4f})')
ax2.fill_between(recall, 0, precision, color='#39C5BB', alpha=0.2)

# Mark the specific point on the curve where our "Best Threshold" sits
# (We find the closest recall/precision values for our chosen threshold)
idx = np.argmin(np.abs(pr_thresholds - best_threshold))
ax2.scatter(recall[idx], precision[idx], color='red', s=100, zorder=5, label='Chosen Operating Point')

ax2.set_title("Precision-Recall Curve\n(The 'Imbalance' Gold Standard)", fontsize=14)
ax2.set_xlabel("Recall (Ability to find Reactives)")
ax2.set_ylabel("Precision (Ability to be correct)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# SHAP

In [ ]:
import shap

# 1. Initialize the Tree Explainer for XGBoost
explainer_xgb = shap.TreeExplainer(best_xgb_model)
shap_vals_xgb = explainer_xgb.shap_values(X_test_processed)

# XGBoost typically returns a single 2D array for binary classification,
# but we add a safety check just in case your specific version returns a list!
if isinstance(shap_vals_xgb, list):
    shap_raw_xgb = shap_vals_xgb[1]
else:
    shap_raw_xgb = shap_vals_xgb

# Put them in a DataFrame for easy grouping
shap_df_xgb = pd.DataFrame(shap_raw_xgb, columns=X_test_processed.columns)
grouped_shap_df_xgb = shap_df_xgb.copy()

# 2. Define the exact prefixes of your One-Hot Encoded columns
categorical_prefixes = ['cat__Sex','cat__Age_Group','cat__Transmission',
                        'cat__Healthcare_Access_Friction','cat__Civil_Status','cat__OFW_Status','cat__Chemsex_Engagement',
                        'cat__Alcohol_Sex_Risk','cat__PrEP_Awareness','cat__Transactional_Sex','cat__STI_BBV_CoInfection_Any']

# 3. Stitch the dummy columns back together!
for prefix in categorical_prefixes:
    # Find all columns that start with this prefix
    dummy_cols = [col for col in X_test_processed.columns if col.startswith(f"{prefix}_")]

    if len(dummy_cols) > 0:
        # Sum the dummy parts to get the total category importance
        grouped_shap_df_xgb[prefix] = shap_df_xgb[dummy_cols].sum(axis=1)
        # Drop the diluted dummy columns
        grouped_shap_df_xgb.drop(columns=dummy_cols, inplace=True)

In [ ]:
# 4. Plot the Global Feature Importance (Miku Teal!)
plt.figure(figsize=(12, 8))
plt.title(f"XGBoost: True Feature Importance (Grouped)", fontsize=14)

shap.summary_plot(
    grouped_shap_df_xgb.values,
    features=grouped_shap_df_xgb,
    feature_names=grouped_shap_df_xgb.columns,
    plot_type="bar",
    color="#39C5BB", #turqoise
    show=False
)

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## TOP 10 INDIVIDUAL FEATURES 

In [ ]:
explainer_xgb = shap.TreeExplainer(best_xgb_model)
shap_vals_xgb = explainer_xgb.shap_values(X_test_processed)

plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_vals_xgb,
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="bar",                        # Beeswarm style
    color="#39C5BB",
    max_display=10,
    show=False
)

# Optional: Adding a subtle grid
plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()


## Beeswarm

In [ ]:
import matplotlib.colors as mcolors
plt.figure(figsize=(10, 6))
miku_cmap = mcolors.LinearSegmentedColormap.from_list("miku_gradient", ["#D1F2F0", "#39C5BB"])
shap.summary_plot(
    shap_vals_xgb,
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="dot",                        # Beeswarm style
    cmap=miku_cmap,
    max_display=10,                         # Top 10 only
    show=False
)

# Optional: Adding a subtle grid
plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()

# SAVING THE MODEL (RUN MO TO YA IF YOU THINK MAGANDA RESULT PARA MASAVE)

In [ ]:
import joblib
model_filename_xgb = 'best_xgb_model.joblib'
joblib.dump(best_xgb_model, model_filename_xgb)

print(f"\nModel successfully saved to {model_filename_xgb}")